### ЗАДАЧА: Расписание занятий и поиск конфликтов

Дан список с расписанием занятий.

Каждая строка файла имеет формат:
день,время,группа,предмет,аудитория

Где:
- день        — день недели (строка, например "Понедельник")
- время       — время начала занятия (строка, например "09:00")
- группа      — название учебной группы
- предмет     — название предмета
- аудитория   — номер аудитории

НЕОБХОДИМО РЕАЛИЗОВАТЬ:

1. Записать список в файл schedule.txt

2. Прочитать файл schedule.txt и загрузить все занятия.

3. Для каждой группы:
   - сформировать список предметов, которые она изучает
   - посчитать общее количество занятий

4. Найти конфликты аудиторий:
   - конфликт — это ситуация, когда в одной и той же аудитории
     в один и тот же день и время назначены занятия
     для разных групп
   - сохранить все такие конфликты в отдельную структуру

5. Для каждого дня недели посчитать количество занятий.

6. Найти день с наибольшим количеством занятий.

7. Записать подробный отчёт в файл schedule_report.txt,
   включая:
   - статистику по группам
   - список конфликтов
   - день с максимальной нагрузкой

In [ ]:
lines = [
    "Понедельник,09:00,Группа1,Математика,101",
    "Понедельник,09:00,Группа2,Физика,101",
    "Понедельник,10:30,Группа1,Физика,102",
    "Понедельник,12:00,Группа3,История,103",
    "Вторник,09:00,Группа1,Информатика,101",
    "Вторник,09:00,Группа2,Математика,102",
    "Вторник,10:30,Группа3,Физика,101",
    "Вторник,12:00,Группа1,История,103",
    "Среда,09:00,Группа2,Информатика,101",
    "Среда,10:30,Группа3,Математика,101",
    "Среда,10:30,Группа1,Физика,101"
]

schedule = []  
# список всех занятий
# каждый элемент может быть словарём с ключами:
# "day", "time", "group", "subject", "room"

group_subjects = {}
# group -> список предметов этой группы

group_lesson_count = {}
# group -> общее количество занятий

room_usage = {}
# (day, time, room) -> список групп
# используется для поиска конфликтов

day_lesson_count = {}
# day -> количество занятий в этот день

conflicts = []
# список найденных конфликтов расписания

# Записываем строки в файл
with open("schedule.txt", "w", encoding="utf-8") as file:
    for line in lines:
        file.write(line + "\n")

# Читаем файл и разбираем строки
with open("schedule.txt", "r", encoding="utf-8") as file:
    for line in file:
        line = line.strip()
        if not line:
            continue

        # Разбираем строку по разделителю
        parts = line.split(",")
        if len(parts) == 5:
            day, time, group, subject, room = parts
            
            # Создаём словарь занятия
            lesson = {
                "day": day,
                "time": time,
                "group": group,
                "subject": subject,
                "room": room
            }
            
            # Добавляем в расписание
            schedule.append(lesson)

# Обрабатываем каждое занятие
for lesson in schedule:
    day = lesson["day"]
    time = lesson["time"]
    group = lesson["group"]
    subject = lesson["subject"]
    room = lesson["room"]

    # 1) Обновляем group_subjects
    if group not in group_subjects:
        group_subjects[group] = []
    if subject not in group_subjects[group]:
        group_subjects[group].append(subject)

    # 2) Обновляем group_lesson_count
    if group not in group_lesson_count:
        group_lesson_count[group] = 0
    group_lesson_count[group] += 1

    # 3) Обновляем day_lesson_count
    if day not in day_lesson_count:
        day_lesson_count[day] = 0
    day_lesson_count[day] += 1

    # 4) Обновляем room_usage
    key = (day, time, room)
    if key not in room_usage:
        room_usage[key] = []
    room_usage[key].append(group)

# Ищем конфликты
for key, groups in room_usage.items():
    # key = (day, time, room)
    # groups = список групп
    if len(groups) > 1:
        # Это конфликт — несколько групп в одной аудитории в одно время
        conflict_info = {
            "day": key[0],
            "time": key[1],
            "room": key[2],
            "groups": groups
        }
        conflicts.append(conflict_info)

# Находим самый загруженный день
busiest_day = None
max_lessons = 0
for day, count in day_lesson_count.items():
    if count > max_lessons:
        max_lessons = count
        busiest_day = day

# Формируем отчёт
with open("schedule_report.txt", "w", encoding="utf-8") as file:
    file.write("ОТЧЁТ ПО РАСПИСАНИЮ\n")
    file.write("=" * 50 + "\n\n")
    
    # Статистика по группам
    file.write("СТАТИСТИКА ПО ГРУППАМ:\n")
    for group in sorted(group_subjects.keys()):
        subjects = ",".join(group_subjects[group])
        count = group_lesson_count[group]
        file.write(f"Группа {group}: {count} занятий ({subjects})\n")
    file.write("\n")
    
    # Список конфликтов
    file.write("КОНФЛИКТЫ РАСПИСАНИЯ:\n")
    if conflicts:
        for conflict in conflicts:
            file.write(
                f"День: {conflict['day']}, Время: {conflict['time']}, "
                f"Аудитория: {conflict['room']} — "
                f"Конфликт между группами: {', '.join(conflict['groups'])}\n"
            )
    else:
        file.write("Конфликтов не обнаружено.\n")
    file.write("\n")
    
    # Самый загруженный день
    file.write("САМЫЙ ЗАГРУЖЕННЫЙ ДЕНЬ:\n")
    if busiest_day:
        file.write(f"{busiest_day} — {max_lessons} занятий\n")
    else:
        file.write("Расписание пустое.\n")

print("Обработка завершена. Отчёт сохранён в 'schedule_report.txt'")
